## 01 — Prepare documents
Builds one clean document per video from the silver table:
filters the business period, keeps the latest row per video,
adds category names, cleans the text and writes `rag_video_documents`.

In [0]:
for key in ["catalog", "silver_schema", "gold_schema", "rag_schema"]:
    dbutils.widgets.text(key, "")

In [0]:
%run ./00_config

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import StringType

from rag_youtube.text_cleaning import clean_text, clean_tags

clean_text_udf = F.udf(clean_text, StringType())
clean_tags_udf = F.udf(clean_tags, StringType())

In [0]:
period = config["source"]["trending_period"]
date_format = config["source"]["trending_date_format"]

silver_df = (
    spark.table(silver_table)
    .withColumn("trending_dt", F.to_date("trending_date", date_format))
    .filter(F.col("trending_dt").between(period["start"], period["end"]))
)

In [0]:
latest_window = Window.partitionBy("video_id").orderBy(F.col("trending_dt").desc(), F.col("views").desc())

latest_df = (
    silver_df
    .withColumn("row_num", F.row_number().over(latest_window))
    .filter(F.col("row_num") == 1)
    .drop("row_num")
)

In [0]:
silver_rows = spark.table(silver_table).count()
in_period_rows = silver_df.count()
documents = latest_df.count()
unique_videos = latest_df.select("video_id").distinct().count()

logger.info("Silver rows: %s, in period: %s, documents: %s", silver_rows, in_period_rows, documents)

if documents != unique_videos:
    raise ValueError(f"Deduplication failed: {documents} rows but {unique_videos} unique videos")

In [0]:
categories_df = spark.table(category_dim_table).select("category_id", "category_name")

enriched_df = (
    latest_df
    .withColumn("category_id", F.col("category_id").cast("int"))
    .join(categories_df, on="category_id", how="left")
    .withColumn("category_name", F.coalesce(F.col("category_name"), F.lit("Unknown")))
)

In [0]:
documents_df = (
    enriched_df
    .withColumn("title_clean", clean_text_udf("title"))
    .withColumn("description_clean", clean_text_udf("description"))
    .withColumn("tags_clean", clean_tags_udf("tags"))
    .withColumn(
        "document_text",
        F.concat_ws(
            "\n",
            F.concat(F.lit("Title: "), F.col("title_clean")),
            F.concat(F.lit("Channel: "), F.col("channel_title")),
            F.concat(F.lit("Category: "), F.col("category_name")),
            F.concat(F.lit("Tags: "), F.col("tags_clean")),
            F.concat(F.lit("Description: "), F.col("description_clean")),
        ),
    )
    .select(
        "video_id",
        F.col("title_clean").alias("title"),
        "channel_title",
        "category_id",
        "category_name",
        F.col("tags_clean").alias("tags"),
        F.col("description_clean").alias("description"),
        F.col("trending_dt").alias("last_trending_date"),
        "views",
        "likes",
        "document_text",
        F.current_timestamp().alias("_prepared_at"),
    )
)

In [0]:
(
    documents_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(documents_table)
)

logger.info("Documents written: %s rows to %s", spark.table(documents_table).count(), documents_table)